Blocks to select an interval of water level data from the eo-tides results and PUV data, interpolate to common timestamps, and various plots. The PUV "water_variation" is corrected against the mean of water depth readings of the PUV sensor.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime
import pandas as pd
import numpy as np


In [ ]:
# Load Data
df_sig = pd.read_csv("sig1000_waves.csv")
df_tide = pd.read_csv("tidal_data_marconi_modelled_2024-10-01-2025-03-31.csv")

# Define interval of interest
start_time = min(df_sig["time"])
end_time = max(df_sig["time"])
df_sig["time"] = pd.to_datetime(df_sig["time"])
df_tide["time"] = pd.to_datetime(df_tide["time"])
df_tide = df_tide[df_tide["time"].between(start_time,end_time)]

# Subtracts mean water depth from PUV sensor water depth to get an approximation for tides
df_sig["water_depth_mean"] = df_sig["water_depth"].mean()
df_sig["water_variation"] = df_sig["water_depth"]-df_sig["water_depth_mean"]


# Interpolate tide values to PUV timestamps
columns = ['FES2022_extrapolated','EOT20','GOT5.5',
       'GOT5.6']
known_times = df_tide['time']
target_times = df_sig['time']
known_seconds = known_times.astype(np.int64)
target_seconds = target_times.astype(np.int64)
df_new = df_sig
for column in columns:
    known_values = df_tide[column]
    interpolated_values = np.interp(target_seconds, known_seconds, known_values)
    df_new[column] = interpolated_values

In [ ]:
# Lineplots of PUV "tidal" signal and modelled tides using FES2022b
fig, ax = plt.subplots(figsize = (12,6))
ax.plot(df_tide["time"],df_tide["FES2022_extrapolated"],label="FES2022b tide") # convert from cm
ax.plot(df_sig["time"],df_sig["water_variation"],label = "Signature 1000 corrected by mean")

# Set daily ticks and format
ax.xaxis.set_major_locator(mdates.DayLocator(interval=2))  # every day
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
plt.setp(ax.get_xticklabels(), rotation=45, ha='right')
ax.legend(loc='lower center', bbox_to_anchor=(0.5, -0.50),ncol=4,fontsize = 16)
ax.set_xlabel("Date")
ax.set_ylabel("Water Level (m)")
ax.set_title("FES2022b vs PUV Data")
plt.tight_layout()
plt.show()


In [ ]:
# Scatterplot of Modelled Tides vs PUV water levels, with regression
fig, ax = plt.subplots(figsize=(9,9))

cmap = plt.colormaps['Spectral'] 

for i in range(len(columns)):
    x = df_new['water_variation']
    y = df_new[columns[i]]
    
    color = cmap(i / (len(columns) - 1)) 
    m, b = np.polyfit(x, y, deg=1)

    y_pred = m * x + b
    ss_res = np.sum((y - y_pred) ** 2)       # Residual sum of squares [1, 2]
    ss_tot = np.sum((y - np.mean(y)) ** 2)   # Total sum of squares [1, 2]
    r_squared = 1 - (ss_res / ss_tot)        # R² Formula [1, 2]
    label_text = f'{columns[i]}: ($R^2$ = {r_squared:.3f})'
    ax.scatter(x, y, color=color, label=f'{columns[i]}',edgecolors="black",linewidths=0.4,alpha=0.8,zorder=i)
    ax.plot(x, m * x + b, color=color, linewidth=1.8, linestyle='--',label=label_text)
plt.xlabel('Marconi PUV Mean Water Level (m)')
plt.ylabel('Modelled Tide Level (m)')
plt.legend()
plt.grid(True,zorder = 0)
plt.show


